# Hybrid: thêm sentiment vào thuật toán gợi ý

Các biến thể (mỗi biến thể chạy trên từng thuật toán nền trong `best_params.json`):

| Nhóm | Ý tưởng |
|---|---|
| A | Baseline, không dùng review. |
| B | Trọng số tương tác theo số sao. |
| C | Trọng số theo sentiment của bình luận, với ba cách chuẩn hóa: `raw`, `rank` (phân vị toàn cục), `user_rel` (so với mức khen trung bình của chính user đó). |
| D | Trộn sao và sentiment. |
| E | Điểm chất lượng của item (theo sao hoặc sentiment) nhân vào điểm gợi ý (chỉ với ItemKNN, vì điểm EASE có thể âm). |
| F | Dùng sentiment như tín hiệu âm: bỏ khỏi lịch sử các tương tác bị chê (sao thấp hoặc sentiment thấp). |

Hai chuẩn hóa `rank` và `user_rel` thêm vào để giải quyết việc sentiment dồn cục về phía cao (khi đó trọng số gần như không đổi và ma trận có trọng số chẳng khác ma trận nhị phân). Trọng số được chia cho trung bình của chính nó trước khi dựng ma trận, để quy mô ma trận Gram không đổi và `lam` của EASE (tune cho ma trận nhị phân) giữ nguyên ý nghĩa; nếu không, biến thể nào có trọng số nhỏ hơn 1 sẽ bị điều chuẩn mạnh hơn baseline.

Mọi bước tính trọng số, điểm item, trung bình theo user chỉ dùng dữ liệu lịch sử (train khi tuning, train+valid khi chấm test).

In [1]:
import sys; sys.path.append("..")
import json
import numpy as np, pandas as pd
from src.config import CATEGORY, DATA, SPLITS, RESULTS, SENT_FILE
from src.models import ItemKNN, EASE, ItemPriorRerank, PopFallback, build_matrix, build_weighted
from src.metrics import evaluate, sort_targets
from src.stats import ndcg_at, paired_bootstrap

SENT_COL = "tf01"            # đổi thành "vader01" để chạy lại với VADER
ALGOS = ("knn", "ease")      # bỏ "ease" nếu muốn chạy nhanh
EPS = 0.05                   # trọng số tối thiểu của một tương tác
POP_EPS = 1e-4
BEST = json.loads((RESULTS / "best_params.json").read_text())
ALGOS = tuple(a for a in ALGOS if a in BEST)
print(CATEGORY, "| thuật toán nền:", ALGOS, "|", BEST)

sent = pd.read_csv(SENT_FILE)
assert SENT_COL in sent.columns, f"sentiment_all chưa có cột {SENT_COL}"

def load(n):
    d = pd.read_csv(SPLITS / f"{n}.csv.gz").merge(
        sent[["user_id", "parent_asin", "has_signal", SENT_COL]],
        on=["user_id", "parent_asin"], how="left", validate="1:1")
    assert d.has_signal.notna().all(), "sentiment_all.csv.gz thiếu review của split: chạy lại sentiment_scoring và report sau khi đổi dữ liệu"
    d["has_signal"] = d.has_signal.astype(bool)
    assert d.loc[d.has_signal, SENT_COL].notna().all()
    d["z_star"] = (d.rating - 1) / 4
    return d

train, valid, test = load("train"), load("valid"), load("test")
allx = pd.concat([train, valid, test])
user2idx = {u: i for i, u in enumerate(allx.user_id.unique())}
item2idx = {p: i for i, p in enumerate(allx.parent_asin.unique())}
X_tr = build_matrix(train, user2idx, item2idx)
X_trva = build_matrix(pd.concat([train, valid]), user2idx, item2idx)

Video_Games | thuật toán nền: ('knn', 'ease') | {'knn': {'k': 2000, 'shrink': 10.0}, 'ease': {'lam': 100.0}}


## 1. Các hàm dựng trọng số và mô hình

In [2]:
def sent_z(h, mode):
    # đưa sentiment về [0, 1]; dòng không có chữ nhận 0.5 (trung tính)
    s = h[SENT_COL].where(h.has_signal)
    if mode == "raw":
        z = s
    elif mode == "rank":
        z = s.rank(pct=True)
    elif mode == "user_rel":
        g, c = s.mean(), 3                               # co kéo về trung bình toàn cục khi user có ít review
        grp = s.groupby(h.user_id)
        mu = (grp.transform("sum") + c * g) / (grp.transform("count") + c)
        z = 0.5 + 0.5 * np.tanh((s - mu) / s.std())
    else:
        raise ValueError(mode)
    return z.fillna(0.5)

def weights(h, cfg):
    if "drop" in cfg:                                    # nhóm F: trọng số 0/1
        if cfg["drop"] == "stars":
            return (h.rating > cfg["thr"]).astype(float).to_numpy()
        z = sent_z(h, "rank")
        thr = np.quantile(z[h.has_signal], cfg["q"])
        return ((~h.has_signal) | (z > thr)).astype(float).to_numpy()
    a, g = cfg["alpha"], cfg["gamma"]                    # alpha = tỉ trọng của số sao
    zt = sent_z(h, cfg["mode"]).to_numpy() if a < 1 else 0.0
    z = a * h.z_star.to_numpy() + (1 - a) * zt
    return EPS + (1 - EPS) * z ** g

def item_prior(h, kind, c=10):
    v = h.z_star if kind == "stars" else sent_z(h, "rank")
    t = pd.DataFrame({"parent_asin": h.parent_asin.to_numpy(), "v": v.to_numpy()})
    g = t.groupby("parent_asin").v.agg(["sum", "count"])
    sm = (g["sum"] + c * t.v.mean()) / (g["count"] + c)   # co kéo về trung bình, item ít review không bị nhiễu
    z = np.zeros(len(item2idx), dtype=np.float32)
    z[g.index.map(item2idx).to_numpy()] = ((sm - sm.mean()) / sm.std()).to_numpy()
    return z

def fit_base(algo, X, weighted):
    if algo == "knn":
        return ItemKNN(BEST["knn"]["k"], BEST["knn"]["shrink"], weighted=weighted).fit(X)
    return EASE(BEST["ease"]["lam"], weighted=weighted).fit(X)

def make_model(cfg, hist, Xbin):
    fam, algo = cfg["family"], cfg["algo"]
    if fam == "A baseline":
        base = fit_base(algo, Xbin, False)
    elif fam.startswith("E prior"):
        base = ItemPriorRerank(fit_base(algo, Xbin, False), item_prior(hist, "stars" if fam.endswith("stars") else "sent"), cfg["lam"])
    else:
        w = weights(hist, cfg)
        Xw = build_weighted(hist, w / w.mean(), user2idx, item2idx)   # trung bình trọng số = 1, giữ quy mô Gram
        base = fit_base(algo, Xw, True)
    return PopFallback(base, Xbin, POP_EPS)             # mọi biến thể (kể cả A) dùng cùng cách phá hòa

def slug(s):
    return "".join(ch if ch.isalnum() else "_" for ch in s).strip("_")

## 2. Tuning trên valid

In [3]:
configs = []
for algo in ALGOS:
    configs.append(dict(algo=algo, family="A baseline"))
    configs += [dict(algo=algo, family="B stars", alpha=1.0, gamma=g) for g in (0.5, 1, 2)]
    configs += [dict(algo=algo, family="C sentiment", alpha=0.0, mode=m, gamma=g)
                for m in ("raw", "rank", "user_rel") for g in (1, 2)]
    configs += [dict(algo=algo, family="D stars+sentiment", alpha=a, mode=m, gamma=1.0)
                for a in (0.25, 0.5, 0.75) for m in ("rank", "user_rel")]
    configs += [dict(algo=algo, family="F drop low stars", drop="stars", thr=t) for t in (2, 3)]
    configs += [dict(algo=algo, family="F drop low sentiment", drop="sent", q=q) for q in (0.05, 0.1, 0.2)]
    if algo == "knn":
        configs += [dict(algo=algo, family=f"E prior {w}", lam=l)
                    for w in ("stars", "sentiment") for l in (0.1, 0.25, 0.5)]
print("số cấu hình:", len(configs))

rows = []
for cfg in configs:
    m, _ = evaluate(make_model(cfg, train, X_tr), X_tr, valid, user2idx, item2idx)
    params = {k: v for k, v in cfg.items() if k not in ("algo", "family")}
    rows.append({"algo": cfg["algo"], "family": cfg["family"], "params": json.dumps(params),
                 "cfg_json": json.dumps(cfg), **m})
val = pd.DataFrame(rows)
display(val[["algo", "family", "params", "HR@10", "NDCG@10", "NDCG@20"]].round(4))
val.drop(columns="cfg_json").to_csv(RESULTS / "hybrid_valid.csv", index=False)

số cấu hình: 48


,algo,family,params,HR@10,NDCG@10,NDCG@20
0,knn,A baseline,{},0.0910,0.0483,0.0599
1,knn,B stars,"{""alpha"": 1.0, ""gamma"": 0.5}",0.0898,0.0481,0.0594
2,knn,B stars,"{""alpha"": 1.0, ""gamma"": 1}",0.0884,0.0475,0.0591
3,knn,B stars,"{""alpha"": 1.0, ""gamma"": 2}",0.0867,0.0467,0.0576
4,knn,C sentiment,"{""alpha"": 0.0, ""mode"": ""raw"", ""gamma"": 1}",0.0868,0.0468,0.0585
5,knn,C sentiment,"{""alpha"": 0.0, ""mode"": ""raw"", ""gamma"": 2}",0.0825,0.0443,0.0550
6,knn,C sentiment,"{""alpha"": 0.0, ""mode"": ""rank"", ""gamma"": 1}",0.0805,0.0432,0.0534
7,knn,C sentiment,"{""alpha"": 0.0, ""mode"": ""rank"", ""gamma"": 2}",0.0621,0.0336,0.0420
8,knn,C sentiment,"{""alpha"": 0.0, ""mode"": ""user_rel"", ""gamma"": 1}",0.0824,0.0440,0.0542
9,knn,C sentiment,"{""alpha"": 0.0, ""mode"": ""user_rel"", ""gamma"": 2}",0.0704,0.0380,0.0469


## 3. Chấm test một lần

Mỗi nhóm dùng cấu hình tốt nhất trên valid. Khác biệt giữa các cấu hình trên valid thường nhỏ hơn nhiễu, nên việc chọn này chỉ là một quy ước; kết luận dựa vào khoảng tin cậy ghép cặp của test.

Mỗi nhóm được so với baseline A của cùng thuật toán nền bằng bootstrap ghép cặp. Riêng C và D còn được so với B (chỉ dùng số sao): đó là phép so sánh trả lời câu hỏi sentiment có thêm gì ngoài số sao hay không. Ngoài khoảng tin cậy 95% còn có khoảng đã hiệu chỉnh Bonferroni cho số nhóm được so sánh trên cùng thuật toán nền.

In [4]:
best_cfg = val.loc[val.groupby(["algo", "family"])["NDCG@10"].idxmax()]
hist = pd.concat([train, valid], ignore_index=True)

ranks, metrics = {}, {}
for _, r in best_cfg.iterrows():
    cfg = json.loads(r.cfg_json)
    m, rk = evaluate(make_model(cfg, hist, X_trva), X_trva, test, user2idx, item2idx)
    key = (cfg["algo"], cfg["family"])
    ranks[key], metrics[key] = rk, (m, r.params)
    np.save(RESULTS / f"ranks_hybrid_{cfg['algo']}_{slug(cfg['family'])}.npy", rk)

rows = []
for algo in ALGOS:
    fams = [f for (a, f) in ranks if a == algo and f != "A baseline"]
    n_cmp = max(len(fams), 1)
    ra = ranks[(algo, "A baseline")]
    rows.append({"algo": algo, "family": "A baseline", "params": "{}",
                 "HR@10": metrics[(algo, "A baseline")][0]["HR@10"],
                 "NDCG@10": metrics[(algo, "A baseline")][0]["NDCG@10"]})
    for f in sorted(fams):
        d, lo, hi = paired_bootstrap(ra, ranks[(algo, f)], 10, "ndcg", alpha=0.05)
        _, lob, hib = paired_bootstrap(ra, ranks[(algo, f)], 10, "ndcg", alpha=0.05 / n_cmp)
        m, params = metrics[(algo, f)]
        rows.append({"algo": algo, "family": f, "params": params, "HR@10": m["HR@10"], "NDCG@10": m["NDCG@10"],
                     "dNDCG@10": d, "lo": lo, "hi": hi, "lo_bonf": lob, "hi_bonf": hib,
                     "khác baseline (95%)": bool(lo > 0 or hi < 0),
                     "khác baseline (Bonferroni)": bool(lob > 0 or hib < 0)})
res = pd.DataFrame(rows)
display(res.round(4))
res.to_csv(RESULTS / "hybrid_test.csv", index=False)

# C và D so với B: sentiment có thêm gì ngoài số sao? (2 phép so sánh mỗi thuật toán nền -> Bonferroni, alpha = 0.05 / 2)
rows_b = []
for algo in ALGOS:
    rb = ranks[(algo, "B stars")]
    for f in ("C sentiment", "D stars+sentiment"):
        d, lo, hi = paired_bootstrap(rb, ranks[(algo, f)], 10, "ndcg", alpha=0.025)
        rows_b.append({"algo": algo, "family": f, "so với": "B stars", "dNDCG@10": d, "lo": lo, "hi": hi,
                       "khác 0": bool(lo > 0 or hi < 0)})
vs_b = pd.DataFrame(rows_b)
display(vs_b.round(4))
vs_b.to_csv(RESULTS / "hybrid_vs_stars_test.csv", index=False)


,algo,family,params,HR@10,NDCG@10,dNDCG@10,lo,hi,lo_bonf,hi_bonf,khác baseline (95%),khác baseline (Bonferroni)
0,knn,A baseline,{},0.0737,0.0386,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,knn,B stars,"{""alpha"": 1.0, ""gamma"": 0.5}",0.0717,0.0377,-0.0009,-0.0017,-0.0001,-0.0019,0.0001,True,False
2,knn,C sentiment,"{""alpha"": 0.0, ""mode"": ""raw"", ""gamma"": 1}",0.0717,0.0376,-0.0010,-0.0020,0.0001,-0.0023,0.0004,False,False
3,knn,D stars+sentiment,"{""alpha"": 0.75, ""mode"": ""rank"", ""gamma"": 1.0}",0.0728,0.0378,-0.0008,-0.0018,0.0002,-0.0021,0.0006,False,False
4,knn,E prior sentiment,"{""lam"": 0.1}",0.0715,0.0374,-0.0011,-0.0021,-0.0002,-0.0025,0.0002,True,False
5,knn,E prior stars,"{""lam"": 0.1}",0.0713,0.0373,-0.0013,-0.0022,-0.0004,-0.0026,-0.0000,True,True
6,knn,F drop low sentiment,"{""drop"": ""sent"", ""q"": 0.05}",0.0718,0.0375,-0.0011,-0.0019,-0.0003,-0.0021,-0.0000,True,True
7,knn,F drop low stars,"{""drop"": ""stars"", ""thr"": 2}",0.0709,0.0370,-0.0015,-0.0025,-0.0006,-0.0028,-0.0003,True,True
8,ease,A baseline,{},0.0806,0.0432,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,ease,B stars,"{""alpha"": 1.0, ""gamma"": 0.5}",0.0794,0.0429,-0.0002,-0.0012,0.0006,-0.0014,0.0009,False,False


,algo,family,so với,dNDCG@10,lo,hi,khác 0
0,knn,C sentiment,B stars,-0.0001,-0.0010,0.0008,False
1,knn,D stars+sentiment,B stars,0.0001,-0.0006,0.0008,False
2,ease,C sentiment,B stars,-0.0004,-0.0014,0.0005,False
3,ease,D stars+sentiment,B stars,0.0002,-0.0005,0.0009,False


## 4. Phân tầng theo đặc điểm của item đích

Câu hỏi: sentiment có giúp ở nhóm mà số sao ít thông tin không? Test được chia theo (a) số sao của item đích, (b) item đích được user khen nhiều hơn hay ít hơn mức họ thường khen (theo sentiment, tính từ lịch sử). Đây là phân tích thăm dò: các tầng nhỏ có khoảng tin cậy rộng và không hiệu chỉnh nhiều phép so sánh.

In [5]:
tt = sort_targets(test)
mu_hist = hist[hist.has_signal].groupby("user_id")[SENT_COL].mean()
rel = tt[SENT_COL].where(tt.has_signal) - tt.user_id.map(mu_hist)
strata = {
    "tất cả": np.ones(len(tt), dtype=bool),
    "đích <= 3 sao": (tt.rating <= 3).to_numpy(),
    "đích >= 4 sao": (tt.rating >= 4).to_numpy(),
    "đích được khen hơn thường lệ": (rel > 0).to_numpy(),
    "đích được khen ít hơn thường lệ": (rel < 0).to_numpy(),
}
rows = []
for algo in ALGOS:
    ra = ranks[(algo, "A baseline")]
    for (a, f), rk in ranks.items():
        if a != algo or f == "A baseline":
            continue
        for sname, mask in strata.items():
            if mask.sum() < 30:
                continue
            d, lo, hi = paired_bootstrap(ra[mask], rk[mask], 10, "ndcg")
            rows.append({"algo": algo, "family": f, "tầng": sname, "n": int(mask.sum()),
                         "NDCG@10 baseline": ndcg_at(ra[mask], 10).mean(),
                         "NDCG@10 biến thể": ndcg_at(rk[mask], 10).mean(),
                         "d": d, "lo": lo, "hi": hi, "khác 0": bool(lo > 0 or hi < 0)})
strata_df = pd.DataFrame(rows)
display(strata_df.round(4))
strata_df.to_csv(RESULTS / "hybrid_strata_test.csv", index=False)

meta = {"category": CATEGORY, "sent_col": SENT_COL, "algos": list(ALGOS), "eps": EPS, "pop_eps": POP_EPS,
        "best_params": BEST, "n_configs": len(configs), "n_test_users": int(len(tt)),
        "generated_at": pd.Timestamp.now().isoformat(timespec="seconds")}
(RESULTS / "hybrid_run_meta.json").write_text(json.dumps(meta, ensure_ascii=False, indent=2), encoding="utf-8")

,algo,family,tầng,n,NDCG@10 baseline,NDCG@10 biến thể,d,lo,hi,khác 0
0,knn,B stars,tất cả,16886,0.0386,0.0377,-0.0009,-0.0017,-0.0001,True
1,knn,B stars,đích <= 3 sao,3582,0.0208,0.0179,-0.0028,-0.0052,-0.0007,True
2,knn,B stars,đích >= 4 sao,13304,0.0433,0.0430,-0.0003,-0.0012,0.0005,False
3,knn,B stars,đích được khen hơn thường lệ,9513,0.0405,0.0401,-0.0004,-0.0014,0.0007,False
4,knn,B stars,đích được khen ít hơn thường lệ,7136,0.0339,0.0324,-0.0016,-0.0030,-0.0003,True
5,knn,C sentiment,tất cả,16886,0.0386,0.0376,-0.0010,-0.0020,0.0001,False
6,knn,C sentiment,đích <= 3 sao,3582,0.0208,0.0171,-0.0037,-0.0061,-0.0014,True
7,knn,C sentiment,đích >= 4 sao,13304,0.0433,0.0431,-0.0003,-0.0014,0.0010,False
8,knn,C sentiment,đích được khen hơn thường lệ,9513,0.0405,0.0403,-0.0002,-0.0016,0.0012,False
9,knn,C sentiment,đích được khen ít hơn thường lệ,7136,0.0339,0.0318,-0.0022,-0.0039,-0.0005,True


335